# TravelFit — Preprocessing Data

Notebook ringkas untuk membersihkan dataset destinasi, menggabungkan rating, membuat fitur C1/C2/C4/C5/C6, dan menyiapkan data K-Means. C3 dihitung saat pengguna memasukkan lokasi asal.

In [16]:
import json
import re
import sys
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/TravelFit')
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'preprocessing'
for folder in (RAW_DIR, PROCESSED_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Environment :', 'Google Colab + Drive' if IN_COLAB else 'Local')
print('Project     :', PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : Google Colab + Drive
Project     : /content/drive/MyDrive/TravelFit


In [17]:
# Unduh hanya jika dataset belum tersedia.
URLS = {
    'destinations': 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_with_id.csv',
    'ratings': 'https://raw.githubusercontent.com/akhiyarwaladi/indonesia_tourism/main/tourism_rating.csv',
}
FILES = {
    'destinations': RAW_DIR / 'tourism_with_id.csv',
    'ratings': RAW_DIR / 'tourism_rating.csv',
}

for name, url in URLS.items():
    if not FILES[name].exists():
        print('Mengunduh:', FILES[name].name)
        urlretrieve(url, FILES[name])

destinations = pd.read_csv(FILES['destinations'])
ratings = pd.read_csv(FILES['ratings'])
print('Destinasi:', destinations.shape, '| Rating:', ratings.shape)
destinations.head(3)

Destinasi: (437, 13) | Rating: (10000, 3)


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Unnamed: 11,Unnamed: 12
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat dengan Monas atau Tugu Monas adalah monumen peringatan setinggi 132 met...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,NaN,1
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tua, berpusat di Alun-Alun Fatahillah, yaitu alun-alun yang ramai dengan...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,NaN,2
2,3,Dunia Fantasi,"Dunia Fantasi atau disebut juga Dufan adalah tempat hiburan yang terletak di kawasan Taman Impian Jaya Ancol, Jakart...",Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,NaN,3


In [18]:
# 1. Bersihkan kolom, teks, tipe data, duplikat, dan baris tidak valid.
def snake_case(value):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(value).strip()).strip('_').lower()

destinations.columns = [snake_case(c) for c in destinations.columns]
ratings.columns = [snake_case(c) for c in ratings.columns]
destinations = destinations.loc[:, ~destinations.columns.str.startswith('unnamed')].copy()
ratings = ratings.loc[:, ~ratings.columns.str.startswith('unnamed')].copy()

for column in ['place_name', 'description', 'category', 'city']:
    destinations[column] = destinations[column].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
for column in ['place_id', 'price', 'rating', 'time_minutes', 'lat', 'long']:
    destinations[column] = pd.to_numeric(destinations[column], errors='coerce')
for column in ['user_id', 'place_id', 'place_ratings']:
    ratings[column] = pd.to_numeric(ratings[column], errors='coerce')

destinations = destinations.drop_duplicates('place_id', keep='first')
destinations = destinations.loc[
    destinations['place_id'].notna()
    & destinations['place_name'].notna()
    & destinations['price'].ge(0)
    & destinations['rating'].between(1, 5)
    & destinations['lat'].between(-90, 90)
    & destinations['long'].between(-180, 180)
].copy()
print('Data valid:', destinations.shape)

Data valid: (437, 11)


In [19]:
# 2. Seragamkan wilayah dan kategori.
CITY_TO_PROVINCE = {
    'Jakarta': 'DKI Jakarta', 'Bandung': 'Jawa Barat',
    'Semarang': 'Jawa Tengah', 'Yogyakarta': 'DI Yogyakarta',
    'Surabaya': 'Jawa Timur',
}
CATEGORY_MAP = {
    'budaya': 'budaya', 'taman hiburan': 'hiburan',
    'cagar alam': 'alam', 'pusat perbelanjaan': 'belanja',
    'tempat ibadah': 'religi', 'bahari': 'bahari',
}
destinations['province'] = destinations['city'].map(CITY_TO_PROVINCE).astype('string')
destinations['category_original'] = destinations['category']
destinations['category_clean'] = destinations['category'].str.casefold().map(CATEGORY_MAP).astype('string')
destinations[['category_original', 'category_clean']].drop_duplicates()

,category_original,category_clean
0,Budaya,budaya
2,Taman Hiburan,hiburan
6,Cagar Alam,alam
8,Bahari,bahari
14,Pusat Perbelanjaan,belanja
21,Tempat Ibadah,religi


In [20]:
# 3. Bersihkan dan agregasikan rating pengguna.
ratings = ratings.loc[
    ratings['user_id'].notna()
    & ratings['place_id'].notna()
    & ratings['place_ratings'].between(1, 5)
].drop_duplicates(['user_id', 'place_id'], keep='last')

rating_summary = ratings.groupby('place_id', as_index=False).agg(
    user_rating_mean=('place_ratings', 'mean'),
    user_rating_count=('place_ratings', 'count'),
    user_rating_std=('place_ratings', 'std'),
)
rating_summary['user_rating_std'] = rating_summary['user_rating_std'].fillna(0)
destinations = destinations.merge(rating_summary, on='place_id', how='left', validate='one_to_one')
destinations['c1_ticket_price'] = destinations['price'].astype(float)
destinations['c2_rating'] = destinations['rating'].astype(float)
destinations[['place_name', 'c1_ticket_price', 'c2_rating', 'user_rating_count']].head()

,place_name,c1_ticket_price,c2_rating,user_rating_count
0,Monumen Nasional,20000.0,4.6,18
1,Kota Tua,0.0,4.6,24
2,Dunia Fantasi,270000.0,4.6,18
3,Taman Mini Indonesia Indah (TMII),10000.0,4.5,21
4,Atlantis Water Adventure,94000.0,4.5,23


In [21]:
# 4. Bentuk fitur fasilitas C4 dan tag aktivitas untuk C6.
# Keduanya masih heuristik dari deskripsi dan perlu verifikasi untuk penelitian final.
FACILITY_KEYWORDS = {
    'toilet': ['toilet', 'kamar mandi', 'wc umum'],
    'parking': ['parkir', 'parking'],
    'food': ['warung', 'restoran', 'rumah makan', 'kafe', 'cafe', 'kuliner'],
    'worship': ['mushola', 'musala', 'masjid', 'tempat ibadah', 'gereja', 'pura', 'vihara'],
    'accessibility': ['disabilitas', 'difabel', 'kursi roda', 'wheelchair', 'aksesibel'],
    'information_center': ['pusat informasi', 'information center', 'layanan informasi'],
}
ACTIVITY_KEYWORDS = {
    'hiking': ['hiking', 'mendaki', 'pendakian', 'trekking'],
    'fotografi': ['fotografi', 'spot foto', 'berfoto', 'pemandangan', 'panorama'],
    'snorkeling': ['snorkeling', 'snorkel'],
    'diving': ['diving', 'menyelam', 'selam'],
    'camping': ['camping', 'berkemah', 'bumi perkemahan'],
    'kuliner': ['kuliner', 'makanan khas', 'jajanan', 'warung', 'restoran'],
    'sejarah': ['sejarah', 'bersejarah', 'peninggalan', 'museum', 'monumen'],
    'budaya': ['budaya', 'tradisi', 'kesenian', 'keraton'],
    'belanja': ['belanja', 'pusat perbelanjaan', 'pasar', 'mal', 'mall'],
    'berenang': ['berenang', 'kolam renang', 'waterpark', 'water park'],
    'edukasi': ['edukasi', 'pendidikan', 'belajar', 'museum'],
    'religi': ['ziarah', 'religi', 'ibadah', 'masjid', 'gereja', 'pura', 'vihara'],
    'rekreasi_keluarga': ['keluarga', 'wahana', 'taman bermain', 'taman hiburan'],
}
CATEGORY_TAGS = {
    'budaya': {'budaya'}, 'belanja': {'belanja'},
    'religi': {'religi'}, 'hiburan': {'rekreasi_keluarga'},
}

def normalize_text(value):
    return re.sub(r'\s+', ' ', str(value).casefold()).strip() if pd.notna(value) else ''

def contains_keyword(text, keyword):
    escaped = re.escape(keyword.casefold()).replace(r'\ ', r'\s+')
    return re.search(rf'(?<!\w){escaped}(?!\w)', text) is not None

description_text = destinations['description'].map(normalize_text)
facility_columns = []
for facility, keywords in FACILITY_KEYWORDS.items():
    column = f'facility_{facility}_mentioned'
    destinations[column] = description_text.map(
        lambda text, keys=keywords: int(any(contains_keyword(text, key) for key in keys))
    )
    facility_columns.append(column)
destinations['c4_facility_score'] = destinations[facility_columns].mean(axis=1)

def extract_activity_tags(row):
    text = normalize_text(f"{row['place_name']} {row['description']}")
    tags = set(CATEGORY_TAGS.get(row['category_clean'], set()))
    for tag, keywords in ACTIVITY_KEYWORDS.items():
        if any(contains_keyword(text, keyword) for keyword in keywords):
            tags.add(tag)
    return '|'.join(sorted(tags))

destinations['activity_tags'] = destinations.apply(extract_activity_tags, axis=1)
destinations['feature_source_note'] = 'C4 dan activity_tags merupakan heuristik deskripsi; perlu verifikasi'
destinations[['place_name', 'c4_facility_score', 'category_clean', 'activity_tags']].head(10)

,place_name,c4_facility_score,category_clean,activity_tags
0,Monumen Nasional,0.0,budaya,budaya|sejarah
1,Kota Tua,0.0,budaya,budaya|edukasi|sejarah
2,Dunia Fantasi,0.0,hiburan,rekreasi_keluarga
3,Taman Mini Indonesia Indah (TMII),0.0,hiburan,budaya|rekreasi_keluarga
4,Atlantis Water Adventure,0.0,hiburan,berenang|rekreasi_keluarga
5,Taman Impian Jaya Ancol,0.0,hiburan,rekreasi_keluarga
6,Kebun Binatang Ragunan,0.0,alam,belanja
7,Ocean Ecopark,0.0,hiburan,edukasi|rekreasi_keluarga
8,Pelabuhan Marina,0.0,bahari,rekreasi_keluarga
9,Pulau Tidung,0.0,bahari,


In [22]:
# 5. Standardisasi fitur numerik dan one-hot encoding kategori untuk K-Means

NUMERIC_FEATURES = [
    "c1_ticket_price",
    "c2_rating",
    "c4_facility_score",
]

numeric_data = destinations[NUMERIC_FEATURES].astype(float)

# Standardisasi Z-score
feature_mean = numeric_data.mean()
feature_scale = numeric_data.std(ddof=0).replace(0, 1)

scaled_data = (numeric_data - feature_mean) / feature_scale
scaled_data.columns = [
    f"{column}_z" for column in NUMERIC_FEATURES
]

# One-hot encoding kategori
category_data = pd.get_dummies(
    destinations["category_clean"],
    prefix="category",
    dtype=int,
)

# Gabungkan identitas destinasi dengan fitur hasil transformasi
kmeans_ready = pd.concat(
    [
        destinations[
            [
                "place_id",
                "place_name",
                "city",
                "province",
                "category_clean",
            ]
        ].reset_index(drop=True),
        scaled_data.reset_index(drop=True),
        category_data.reset_index(drop=True),
    ],
    axis=1,
)

print("Dataset bersih:", destinations.shape)
print("Siap K-Means :", kmeans_ready.shape)

kmeans_ready.head()

Dataset bersih: (437, 28)
Siap K-Means : (437, 14)


,place_id,place_name,city,province,category_clean,c1_ticket_price_z,c2_rating_z,c4_facility_score_z,category_alam,category_bahari,category_belanja,category_budaya,category_hiburan,category_religi
0,1,Monumen Nasional,Jakarta,DKI Jakarta,budaya,-0.070094,0.754544,-0.43791,0,0,0,1,0,0
1,2,Kota Tua,Jakarta,DKI Jakarta,budaya,-0.371434,0.754544,-0.43791,0,0,0,1,0,0
2,3,Dunia Fantasi,Jakarta,DKI Jakarta,hiburan,3.696651,0.754544,-0.43791,0,0,0,0,1,0
3,4,Taman Mini Indonesia Indah (TMII),Jakarta,DKI Jakarta,hiburan,-0.220764,0.274579,-0.43791,0,0,0,0,1,0
4,5,Atlantis Water Adventure,Jakarta,DKI Jakarta,hiburan,1.044862,0.274579,-0.43791,0,0,0,0,1,0


In [23]:
# 6. Simpan hasil preprocessing.
destinations['source_dataset'] = 'Indonesia Tourism Destination'
destinations['source_url'] = URLS['destinations']

destinations_path = PROCESSED_DIR / 'destinations_clean.csv'
kmeans_path = PROCESSED_DIR / 'destinations_kmeans_ready.csv'
ratings_path = PROCESSED_DIR / 'ratings_aggregated.csv'
summary_path = REPORT_DIR / 'preprocessing_summary.json'

destinations.to_csv(destinations_path, index=False, encoding='utf-8-sig')
kmeans_ready.to_csv(kmeans_path, index=False, encoding='utf-8-sig')
rating_summary.to_csv(ratings_path, index=False, encoding='utf-8-sig')

summary = {
    'destination_rows': int(len(destinations)),
    'rating_rows_valid': int(len(ratings)),
    'cities': sorted(destinations['city'].dropna().unique().tolist()),
    'categories': sorted(destinations['category_clean'].dropna().unique().tolist()),
    'destinations_with_facility_evidence': int(destinations['c4_facility_score'].gt(0).sum()),
    'destinations_with_activity_tags': int(destinations['activity_tags'].ne('').sum()),
    'scaler_mean': feature_mean.to_dict(),
    'scaler_scale': feature_scale.to_dict(),
}
with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

assert destinations['place_id'].is_unique
assert destinations['c1_ticket_price'].ge(0).all()
assert destinations['c2_rating'].between(1, 5).all()
assert destinations['c4_facility_score'].between(0, 1).all()
assert not kmeans_ready.isna().any().any()

print('Preprocessing selesai.')
print('-', destinations_path)
print('-', kmeans_path)
print('-', ratings_path)
print('-', summary_path)

Preprocessing selesai.
- /content/drive/MyDrive/TravelFit/data/processed/destinations_clean.csv
- /content/drive/MyDrive/TravelFit/data/processed/destinations_kmeans_ready.csv
- /content/drive/MyDrive/TravelFit/data/processed/ratings_aggregated.csv
- /content/drive/MyDrive/TravelFit/reports/preprocessing/preprocessing_summary.json


## Output

- `destinations_clean.csv`: data destinasi bersih dan fitur awal.
- `destinations_kmeans_ready.csv`: data terstandardisasi untuk K-Means.
- `ratings_aggregated.csv`: ringkasan rating per destinasi.
- `preprocessing_summary.json`: ringkasan preprocessing.

C3 jarak, C5 kecocokan kategori, dan C6 Jaccard pengguna dihitung pada tahap modeling setelah data survei tersedia.